In [ ]:
from pathlib import Path
import pandas as pd
import pymzml
import warnings
import io
import sys
from contextlib import redirect_stdout

PROJECT_ROOT = Path("../..").resolve()

MZML_DIR = PROJECT_ROOT / "data" / "raw" / "mzML"
OUT_DIR = PROJECT_ROOT / "results" / "runs" /"runs_05_01_1"/"warning_mzML"
OUT_DIR.mkdir(parents=True, exist_ok=True)

included = []
unreadable = []
no_index_files = []
other_warning_files = []

mzml_files = sorted(list(MZML_DIR.glob("*.mzML")) + list(MZML_DIR.glob("*.mzml")))

print("Project root:", PROJECT_ROOT)
print("mzML dir:", MZML_DIR)
print(f"Found {len(mzml_files)} mzML files.")

for fp in mzml_files:
    f = io.StringIO()

    try:
        with redirect_stdout(f):
            run = pymzml.run.Reader(str(fp))

            for i, spec in enumerate(run):
                if i >= 5:
                    break

        output = f.getvalue()

        if "Not index found" in output:
            no_index_files.append(fp.name)
            print(f"[NO_INDEX] {fp.name}")
        else:
            print(f"[OK]       {fp.name}")

    except Exception as e:
        print(f"[FAIL]     {fp.name} | {repr(e)}")

pd.DataFrame(included + unreadable).to_csv(
    OUT_DIR / "mzml_readability_report.csv",
    index=False
)

pd.DataFrame(other_warning_files).to_csv(
    OUT_DIR / "other_warning_files.csv",
    index=False
)

(OUT_DIR / "included_files.txt").write_text(
    "\n".join(x["file"] for x in included) + "\n",
    encoding="utf-8"
)

(OUT_DIR / "unreadable_files.txt").write_text(
    "\n".join(x["file"] for x in unreadable) + "\n",
    encoding="utf-8"
)

(OUT_DIR / "no_index_files.txt").write_text(
    "\n".join(no_index_files) + "\n",
    encoding="utf-8"
)

print("\nNo-index files")
print("--------------")
for f in no_index_files:
    print(f)

print("\nSummary")
print("-------")
print(f"Found        : {len(mzml_files)}")
print(f"Readable     : {len(included)}")
print(f"No index     : {len(no_index_files)}")
print(f"Unreadable   : {len(unreadable)}")
print("Saved:")
print(OUT_DIR / "included_files.txt")
print(OUT_DIR / "unreadable_files.txt")
print(OUT_DIR / "no_index_files.txt")
print(OUT_DIR / "mzml_readability_report.csv")